# 01 — Del prompt al agente

**Bloque 1, 40 minutos.** Cinco demostraciones, cada una sobre una limitación
del modelo.

El orden no es casual. Cada sección deja a la vista una limitación que la
siguiente idea resuelve, hasta llegar a la única que el prompt no puede
resolver. Ahí aparece la palabra *agente*.

Este notebook se corre entero de arriba abajo y tarda unos diez minutos, salvo
la sección 2, que mide la ventana de contexto y se lleva un minuto más.



<img src="imagenes/01-cerebro-en-frasco.png" width="560">

**Un modelo de lenguaje sabe muchísimo y no alcanza nada.** No tiene reloj, no
tiene internet, no tiene dónde guardar lo que le dijiste y no ejecuta nada. Aun
así contesta con seguridad.

Todo lo que *parece* ser, sin embargo, eso lo pone el programa de alrededor. Este
bloque demuestra, sección por sección, cada una de esas limitaciones y cómo se
resuelve.

In [9]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib

# Recargar agentlib ANTES que lo que depende de el. reload() NO es recursivo:
# si agentlib quedo cacheado de una version anterior, recargar el modulo de
# arriba falla con un "ImportError: cannot import name ... from 'agentlib'"
# que parece del modulo de arriba y en realidad es del paquete de abajo.
#
# Sin esto, basta con que alguien actualice el repo con el kernel abierto para
# que la primera celda falle con un error que no menciona la causa.
import agentlib
import agentlib.ollama
import agentlib.traza
for _modulo in (agentlib.ollama, agentlib.traza, agentlib):
    importlib.reload(_modulo)

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")

from agentlib import (base_url, chat, modelo_configurado, normalizar_argumentos,
                      streaming, ultima_medicion)

# A donde se habla y que modelo se pide salen del .env de la raiz, nunca de
# una celda: asi el mismo notebook corre contra tu Ollama, contra el del
# instructor o contra el servidor del Tec sin editar nada.
URL, MODELO = base_url(), modelo_configurado()
print(f"hablando con {URL}  |  modelo: {MODELO}")

  .env   : c:\Users\Mario\Documents\desarrollo\tec\agenticai\.env  (19 claves)
hablando con http://10.49.12.54:11434  |  modelo: qwen3.5:9b


## 1. Un modelo de lenguaje predice el siguiente token

Nada más que eso. Recibe una lista de mensajes, calcula qué token es más
probable que siga, lo emite y vuelve a empezar con ese token incluido. Repite
hasta emitir el token que marca el final.

Todo lo que *parece* ser más que eso —que recuerde, que consulte, que haga— lo
pone tu programa alrededor. Las secciones 2 a 5 lo comprueban una por una.

Las dos celdas siguientes hacen visible el mecanismo. La primera muestra **que
la respuesta llega de a pedazos**; la segunda, **que los pedazos no son
palabras**. Tardan unos segundos cada una.

In [10]:
# Vamos a la petición cruda, sin agentlib, para ver exactamente qué manda
# Ollama. `stream: True` le dice que no espere a terminar: que mande cada token
# en cuanto lo genere.
import json
import re
import time

import requests

query = "Cuenta del 1 al 10."

CUERPO = {
    "model": MODELO,
    "messages": [{"role": "user", "content": query}],
    "stream": True,          # <- para Ollama: mandá cada token, no esperes
    "think": False,
    "options": {"temperature": 0},
}

crudas = []
t0 = time.time()
# stream=True otra vez, pero este es para requests: no descargues el cuerpo
# completo antes de devolverme el control.
with requests.post(f"{URL}/api/chat", json=CUERPO,
                   stream=True, timeout=180) as r:
    print(f"Content-Type: {r.headers.get('Content-Type')}")
    print(f"Transfer-Encoding: {r.headers.get('Transfer-Encoding')}\n")
    for linea in r.iter_lines():
        if linea:
            crudas.append((round((time.time() - t0) * 1000), linea.decode("utf-8")))

print("=" * 72)
print("0. Esto es el query que mandamos")
print(f"  {query!r}")
print(f"  {len(query)} caracteres, {len(query.encode('utf-8'))} bytes")
print(f" esto es el cuerpo JSON que mandamos a Ollama:\n {json.dumps(CUERPO, indent=2)}")

# ---------------------------------------------------------------------------
print("=" * 72)
print("1. UNA LINEA, exactamente como llega por el socket")
print("=" * 72)
print(crudas[0][1])
print("\nEl token está en message.content. Lo demás es envoltura.")

# ---------------------------------------------------------------------------
print("\n" + "=" * 72)
print("2. TODAS: el JSON que llega, al lado de lo que extraemos")
print("=" * 72)
print(f"  {'ms':>5}  {'lo que llega (prefijo constante eliminado)':<70}  content")
print(f"  {'-' * 5}  {'-' * 70}  {'-' * 7}")

for ms, cruda in crudas:
    d = json.loads(cruda)
    if d.get("done"):
        continue                      # la última va aparte, abajo
    # Elidimos el prefijo, idéntico en todas. Lo que interesa está al final.
    corta = re.sub(r'^\{"model":.*?"created_at":".*?",', "{..., ", cruda)
    print(f"  {ms:5}  {corta:<70}  {d['message']['content']!r}")

# ---------------------------------------------------------------------------
print("\n" + "=" * 72)
print("3. LA ULTIMA es distinta: content vacío, y trae las estadísticas")
print("=" * 72)
# La linea cruda, exactamente como llega, igual que en la seccion 1.
print(crudas[-1][1])
print()

ultima = json.loads(crudas[-1][1])
print("  y desglosada:")
print(f"  message.content : {ultima['message']['content']!r}   <- vacío")
print(f"  done            : {ultima['done']}")
print(f"  done_reason     : {ultima.get('done_reason')!r}")
for k in ("prompt_eval_count", "eval_count"):
    print(f"  {k:<16}: {ultima[k]} tokens")
for k in ("prompt_eval_duration", "eval_duration", "total_duration"):
    print(f"  {k:<20}: {ultima[k] / 1e9:.2f} s")

print("\n  Por eso la función tiene `if pedazo:` — sin esa guarda entregaría")
print("  una cadena vacía al final. Y de `prompt_eval_count` sale la medición")
print("  del costo del español, más abajo.")

# ---------------------------------------------------------------------------
pegado = "".join(json.loads(c)["message"]["content"] for _, c in crudas)
print(f"\n  {len(crudas)} líneas. Pegando todos los content: {pegado}")
print("  Eso es lo único que normalmente verías.")

Content-Type: application/x-ndjson
Transfer-Encoding: chunked

0. Esto es el query que mandamos
  'Cuenta del 1 al 10.'
  19 caracteres, 19 bytes
 esto es el cuerpo JSON que mandamos a Ollama:
 {
  "model": "qwen3.5:9b",
  "messages": [
    {
      "role": "user",
      "content": "Cuenta del 1 al 10."
    }
  ],
  "stream": true,
  "think": false,
  "options": {
    "temperature": 0
  }
}
1. UNA LINEA, exactamente como llega por el socket
{"model":"qwen3.5:9b","created_at":"2026-11-17T19:37:01.880890793Z","message":{"role":"assistant","content":"1"},"done":false}

El token está en message.content. Lo demás es envoltura.

2. TODAS: el JSON que llega, al lado de lo que extraemos
     ms  lo que llega (prefijo constante eliminado)                              content
  -----  ----------------------------------------------------------------------  -------
   2475  {..., "message":{"role":"assistant","content":"1"},"done":false}        '1'
   2481  {..., "message":{"role":"assistant","cont

### Los pedazos no son palabras

En la tabla de arriba, cada renglón es un fragmento del JSON que mandó Ollama y
el token que salió de él. Mira **qué** tokens son:

- La coma llegó **sola**, en su propio fragmento.
- El espacio viaja **pegado** al número que sigue: `"content":" 2"`, no `"2"`.
- El **10 llegó en dos fragmentos**: primero `" 1"`, después `"0"`. Para el
  modelo, "10" no es una cosa.

Eso es un token: ni una letra ni una palabra, sino el pedazo en que *este*
modelo parte *este* texto. Con una palabra larga se ve mejor todavía.

In [11]:
# Una palabra rara, que el modelo no tiene como una sola pieza.
pedazos = list(streaming([{"role": "user",
                           "content": "Escribe solo la palabra: paralelepípedo"}]))

print("el modelo escribió:", "".join(pedazos))
print("y lo hizo en estos pedazos:")
print("  ", pedazos)
print(f"longitud total: {len(pedazos)} con {sum(len(p) for p in pedazos)} caracteres"  )
print()
print("Por eso se cobra y se mide en tokens, no en palabras: son la unidad real.")

el modelo escribió: paralelepípedo
y lo hizo en estos pedazos:
   ['par', 'ale', 'lep', 'í', 'pedo']
longitud total: 5 con 14 caracteres

Por eso se cobra y se mide en tokens, no en palabras: son la unidad real.


**Fíjate dónde cayeron los cortes**, sobre todo en la vocal acentuada: la `í`
suele llegar como un pedazo propio.

Eso tiene una consecuencia práctica: **el español cuesta más tokens que el
inglés.** Los modelos se entrenan con mucho más texto en inglés, así que su
vocabulario tiene piezas grandes para el inglés y piezas más chicas para todo
lo demás.

Medido con este mismo modelo sobre cinco frases equivalentes: **1.28 tokens en
español por cada token en inglés**, con casos que van de 1.00 a 1.78. La
diferencia crece con los acentos y los signos de apertura. `¿Cuántas unidades
hay en el almacén de la cafetería?` cuesta 16 tokens contra los 9 de su
traducción.

Si alguna vez pagas por tokens, ahí está de dónde sale la factura.

La celda siguiente mide dos de esas frases en tu propia máquina. Tarda unos
segundos.

In [12]:
# Midelo tú mismo. `prompt_eval_count` es cuántos tokens leyó el modelo.
import requests

def tokens(texto):
    r = requests.post(f"{URL}/api/chat", json={
        "model": MODELO,
        "messages": [{"role": "user", "content": texto}],
        "stream": False, "think": False,
        "options": {"temperature": 0, "num_predict": 1},
    }, timeout=180).json()
    return r["prompt_eval_count"]

# La plantilla de chat se cobra igual en los dos idiomas, así que la restamos.
base = tokens("")

es = "¿Cuántas unidades hay en el almacén de la cafetería?"
en = "How many units are in the cafeteria warehouse?"
a, b = tokens(es) - base, tokens(en) - base

print(f"  plantilla de chat : {base} tokens (se cobra siempre)")
print(f"  español           : {a:3} tokens   {es}")
print(f"  inglés            : {b:3} tokens   {en}")
print(f"\n  el español cuesta {a/b:.2f}x en este caso")

  plantilla de chat : 12 tokens (se cobra siempre)
  español           :  16 tokens   ¿Cuántas unidades hay en el almacén de la cafetería?
  inglés            :   9 tokens   How many units are in the cafeteria warehouse?

  el español cuesta 1.78x en este caso


## 2. El modelo no tiene memoria

Las dos celdas siguientes hacen la misma pregunta en dos llamadas separadas. En
la segunda, el modelo no tiene forma de saber de qué se habló en la primera.

**La ventana de contexto es la única memoria que existe.** Lo que no está en la
lista de mensajes, no existe. Cuando más adelante digamos que un agente
"recuerda", lo que ocurre es que el programa le reenvía el historial completo en
cada llamada, y lo paga en tokens.

<img src="imagenes/02-pergamino-trituradora.png" width="560">


In [13]:
print(chat([{"role": "user", "content": "Me llamo Mario."}])["content"])


¡Hola, Mario! Es un gusto conocerte. ¿En qué puedo ayudarte hoy?


In [14]:
print(chat([{"role": "user", "content": "¿Cómo me llamo?"}])["content"])

No tengo forma de saber tu nombre, ya que soy una inteligencia artificial y no puedo acceder a tu identidad personal ni a la información privada que tengas fuera de esta conversación.

Si quieres contármelo, ¡por favor dímelo! De lo contrario, puedes llamarme como quieras o simplemente usar mi nombre de modelo.


In [15]:
# Con el historial, sí. No es memoria: es reenviar.
print(chat([
    {"role": "user", "content": "Me llamo Mario."},
    {"role": "assistant", "content": "¡Hola, Mario! Es un gusto conocerte. ¿En qué puedo ayudarte hoy?"},
    {"role": "user", "content": "¿Cómo me llamo?"},
])["content"])

Te llamas **Mario**. ¡Qué nombre tan clásico y enérgico!


### Cuánto cabe en la ventana

Si la ventana es la única memoria que hay, conviene saber de qué tamaño es. Y
ahí hay **tres números distintos**, no uno:

| | de dónde sale |
|---|---|
| lo que **declara el archivo** del modelo | `/api/show` |
| con cuánto **se cargó la instancia** que te atiende | `/api/ps` |
| lo que **pide cada llamada**, si lo pide | la opción `num_ctx` |

El que manda es el tercero cuando está puesto, y el segundo cuando no. El
primero no manda nunca: es el techo del archivo, no lo que te dieron.

**La celda fija el suyo en 4 096** en lugar de usar el del servidor, y eso no es
un detalle de implementación. Con 4 096 el experimento se ve; con los 262 144
que puede tener un servidor grande, no se ve nada. Un experimento cuyo resultado
depende de en qué máquina lo corras no demuestra nada.

**El experimento.** Pone una clave secreta en el primer mensaje, la empuja hacia
atrás con relleno y al final pregunta por ella. Lo único que cambia entre
vueltas es cuánto relleno hay en medio. Al final averigua por qué falla la
primera vuelta que falla, que es la parte que no se adivina.

Vas a ver una tabla de seis filas: las cuatro primeras recuperan la clave, las
dos últimas no. **Tarda menos de un minuto.**

Las dos cifras del servidor también se consultan a mano, y sirve igual para
comprobar que Ollama está vivo antes de correr nada:

```bash
curl $OLLAMA/api/ps      # la ventana de la instancia cargada
curl $OLLAMA/api/tags    # que modelos hay descargados
```

**`$OLLAMA` no es `localhost` para todo el mundo.** Es la dirección que imprimió
la celda de configuración al inicio de este notebook, y sale de
`OLLAMA_DESTINO` en el `.env` de la raíz: en tu máquina es `localhost`, en el
aula la laptop del instructor, y en el Tec una IP del campus. Copia de ahí la
que te tocó.

> En PowerShell usa `curl.exe`, no `curl`: ahí `curl` es un alias de
> `Invoke-WebRequest` y no acepta las mismas banderas. Si no responde ninguna de
> las dos, Ollama no está arriba **en esa dirección**, que no es lo mismo que
> no estar arriba.

> **Y desde WSL, `localhost` es la máquina virtual de WSL**, no Windows. Si
> Ollama corre en tu Windows, desde WSL no siempre lo alcanzas por ahí; usa la
> dirección del `.env`.
>
> Si los comandos **se cuelgan sin error** en WSL o en Linux, revisa el MTU. Una
> VPN suele bajarlo, y entonces las respuestas chicas pasan y las de 4 KB no.
> Se ajusta con `sudo ip link set dev eth0 mtu 1343` y se vuelve a probar.
>
> El síntoma distingue las dos causas: con la dirección equivocada falla
> **todo**, y con el MTU chico `/api/ps` responde y `/api/tags` se cuelga.


In [17]:
# Experimento: donde esta el borde de la ventana, y que pasa al cruzarlo.
import requests

VENTANA = 4096   # la fijamos nosotros, en cada llamada. Ver la nota de abajo.

# 1. Las dos cifras que NO son la misma. La ficha del modelo dice cuanto
#    aguanta el archivo; /api/ps, con cuanto se cargo esta instancia.
ficha = requests.post(f"{URL}/api/show", json={"model": MODELO}, timeout=60).json()
anunciado = next(v for k, v in ficha["model_info"].items()
                 if k.endswith("context_length"))
cargados = requests.get(f"{URL}/api/ps", timeout=30).json()["models"]
del_servidor = cargados[0].get("context_length") if cargados else None

print(f"  modelo                       : {MODELO}")
print(f"  la ficha del modelo anuncia  : {anunciado:>7,} tokens")
if del_servidor:
    print(f"  la instancia cargada usa     : {del_servidor:>7,} tokens")
print(f"  y este experimento pide      : {VENTANA:>7,} tokens en cada llamada")
print()
print(f"  Pedimos la ventana a proposito en vez de usar la del servidor: con")
print(f"  4096 esto se ve, y con 262144 no se ve nada. Un experimento que da")
print(f"  distinto segun la maquina no demuestra nada.")

# 2. La prueba. Una clave al PRINCIPIO, relleno en medio, la pregunta al final.
CLAVE = "MANZANA-7"

def conversacion(renglones):
    relleno = " ".join(f"Renglon {i} del inventario semanal." for i in range(renglones))
    return [
        {"role": "user", "content": f"La clave secreta es {CLAVE}. Recuerdala."},
        {"role": "assistant", "content": "Entendido, la recordare."},
        {"role": "user", "content": relleno},
        {"role": "assistant", "content": "Anotado."},
        {"role": "user", "content": "Cual es la clave secreta? Responde solo con la clave."},
    ]

print(f"\n  {'relleno':>7}  {'leidos':>8} {'de':>7}   {'vio la clave?':<14} que contesto")
print(f"  {'-'*7}  {'-'*8} {'-'*7}   {'-'*14} {'-'*30}")
primera_falla = None
for renglones in (30, 120, 250, 330, 345, 360):
    respuesta = chat(conversacion(renglones), num_ctx=VENTANA)
    leidos = ultima_medicion()["tokens_enviados"]
    la_vio = CLAVE in respuesta["content"]
    if not la_vio and primera_falla is None:
        primera_falla = renglones
    print(f"  {renglones:>7}  {leidos:>8,} {VENTANA:>7,}   "
          f"{('SI' if la_vio else 'NO'):<14} {respuesta['content'].strip()[:30]!r}")

# 3. La pregunta obvia: la fila que fallo leyo MENOS tokens que la ventana,
#    asi que cabia. ?Por que fallo?
#
#    Porque ese numero no es lo que mandamos: es lo que SOBREVIVIO. Y para
#    medir lo que mandamos hay que preguntarselo a una ventana que lo aguante,
#    que es lo unico que cambia entre estas dos llamadas.
if primera_falla is None:
    print(f"\n  Ningun escalon fallo. Sube el relleno hasta pasar de {VENTANA:,}.")
else:
    print(f"\n  la conversacion de la fila que fallo (relleno {primera_falla}), con dos ventanas:")
    for num_ctx in (VENTANA, VENTANA * 2):
        r = chat(conversacion(primera_falla), num_ctx=num_ctx)
        print(f"    num_ctx {num_ctx:>6,} -> leyo {ultima_medicion()['tokens_enviados']:>6,} tokens   "
              f"clave={'SI' if CLAVE in r['content'] else 'NO'}   {r['content'].strip()[:28]!r}")

    # 4. Y que fue, exactamente, lo que se tiro.
    chat(conversacion(primera_falla)[1:], num_ctx=VENTANA * 2)
    sin_cabeza = ultima_medicion()["tokens_enviados"]
    print(f"\n  la misma conversacion SIN su primer mensaje : {sin_cabeza:,} tokens")
    print(f"  y eso es exactamente lo que leyo con la ventana de {VENTANA:,}.")
    print(f"  No recorto tokens: tiro el PRIMER MENSAJE entero -el de la clave-")
    print(f"  porque es la unidad con la que recorta. Y no aviso de nada.")


  modelo                       : qwen3.5:9b
  la ficha del modelo anuncia  : 262,144 tokens
  la instancia cargada usa     :   8,192 tokens
  y este experimento pide      :   4,096 tokens en cada llamada

  Pedimos la ventana a proposito en vez de usar la del servidor: con
  4096 esto se ve, y con 262144 no se ve nada. Un experimento que da
  distinto segun la maquina no demuestra nada.

  relleno    leidos      de   vio la clave?  que contesto
  -------  -------- -------   -------------- ------------------------------
       30       393   4,096   SI             'MANZANA-7'
      120     1,403   4,096   SI             'MANZANA-7'
      250     2,963   4,096   SI             'MANZANA-7'
      330     3,923   4,096   SI             'MANZANA-7'
      345     4,083   4,096   NO             'No tengo una clave secreta.'
      360        36   4,096   NO             'No tengo una clave secreta.'

  la conversacion de la fila que fallo (relleno 345), con dos ventanas:
    num_ctx  4,096 -> le

### Por qué falla una vuelta que "cabía"

La quinta fila leyó 4 083 tokens de una ventana de 4 096 y aun así perdió la
clave. La respuesta es que **no cabía**. Los pasos 3 y 4 de la celda lo miden:

| | tokens |
|---|---|
| lo que la celda **mandó** de verdad (medido con `num_ctx` al doble) | **4 103** |
| la ventana | 4 096 |
| lo que el modelo **leyó** | 4 083 |
| esa misma conversación **sin su primer mensaje** | **4 083** |

Las dos últimas filas son el mismo número. Ollama **descartó el primer mensaje
entero** —el que decía *"La clave secreta es MANZANA-7"*— y leyó lo demás. El
prompt se pasó por siete tokens y el recorte costó veinte, porque Ollama no
recorta tokens: recorta mensajes completos, empezando por el más viejo.

De ahí sale la regla práctica:

> **`tokens_enviados` no avisa de un desborde.** Por construcción no puede
> superar la ventana: cuenta lo que el modelo leyó, no lo que se le mandó.
> Compararlo con 4 096 para detectar desbordes no funciona, porque siempre dirá
> que cabe.

La última fila, la de 36 tokens, es el mismo mecanismo un paso más allá. Ahí
hacían falta unos 67 tokens de recorte, más de lo que pesan los dos primeros
mensajes juntos, así que el siguiente en la fila era el mensaje de relleno.
Descartarlo deja **los dos últimos mensajes: 36 tokens** de una conversación de
4 163. El modelo respondió sobre una conversación de la que leyó el 0.9%.

En las dos filas Ollama devolvió `done_reason: stop`, igual que en cualquier
llamada normal. No hay excepción que atrapar ni campo que revisar.

**Cómo se detecta, entonces:** midiendo el prompt antes de mandarlo, o mandando
el mismo prompt a una ventana más ancha y comparando, que es lo que hace el paso
3 de la celda.

Esta es la trituradora del dibujo, medida. Y no es una curiosidad de
laboratorio: el Experimentador del [notebook 07](07_nas_agente.ipynb) llega a
**4 337 tokens** en su última vuelta barriendo 18 predictores. Contra un Ollama
con la ventana por omisión —4 096— esa vuelta **ya no cabe**: perdería el primer
mensaje de su conversación, donde están sus primeras mediciones, y las seguiría
reportando como si las tuviera. Ahí corre contra un servidor con la ventana
grande y por eso no pasa; en una laptop, pasaría.

> **La ventana se puede agrandar**, con `num_ctx` por petición o con
> `OLLAMA_CONTEXT_LENGTH` en el servidor. Pero el KV cache sale de la VRAM, y en
> la máquina donde se midió esto quedan ~1.3 GB libres con el modelo cargado.
> Hasta dónde se puede subir, y qué cuesta, está medido en el
> [notebook 08](08_ventana_y_memoria.ipynb).

## 3. El system prompt manda

La celda siguiente hace la misma pregunta del usuario con dos system prompts
distintos. La diferencia entre las dos respuestas no la pone el modelo: la pone
quien escribe el system prompt. Es decir, tú.

In [18]:
for identidad in [
    "Eres un profesor de secundaria. Responde en una oración simple.",
    "Eres un investigador. Responde con precisión técnica.",
]:
    r = chat([{"role": "system", "content": identidad},
              {"role": "user", "content": "¿Qué es un LLM?"}])
    print(f"[{identidad}...]\n{r['content']}")
    print("=" * 72)


[Eres un profesor de secundaria. Responde en una oración simple....]
Un LLM es un modelo de lenguaje grande, que es un tipo de inteligencia artificial diseñado para entender y generar texto humano.
[Eres un investigador. Responde con precisión técnica....]
Como investigador, defino a un **LLM** (Large Language Model) o Modelo de Lenguaje Grande como una arquitectura de inteligencia artificial basada en redes neuronales profundas, específicamente diseñada para comprender, generar y manipular lenguaje humano con un alto grado de precisión contextual.

A continuación, detallo los componentes técnicos y funcionales que definen esta categoría:

### 1. Fundamentos Arquitectónicos
La mayoría de los LLMs modernos se basan en la arquitectura de **Transformers**, introducida por Vaswani et al. (2017). Esta estructura permite procesar secuencias de texto de longitud variable mediante dos mecanismos clave:
*   **Atención Multi-cabeza (Multi-Head Attention):** Permite al modelo ponderar la relevanc

## 4. El modelo no es determinista

La misma entrada puede dar salidas distintas. De ahí sale una consecuencia que
el bloque 5 muestra: **un agente no se prueba como se prueba una
función.** Comparar cadenas no sirve.

Demostrarlo tiene una trampa, y vale la pena caer en ella a propósito porque
enseña más que el resultado limpio: **con una respuesta de una sola palabra casi
no se ve.**

Medido con 8 muestras por celda, contando cuántas salidas *distintas* hubo:

| lo que se pide | temp 0 | temp 0.8 | temp 1.5 |
|---|---|---|---|
| "di una fruta" | 1/8 | 2/8 | 3/8 |
| "di un número entre 1 y 100" | 1/8 | 4/8 | 5/8 |
| "inventa un nombre de cafetería" | 1/8 | **8/8** | **8/8** |
| "escribe una oración sobre el mar" | 1/8 | **8/8** | **8/8** |

Una palabra es **una** decisión de muestreo, y si un candidato domina —"Manzana"
lo hace— casi siempre gana. Una oración son unas doce decisiones, y las
divergencias se acumulan: basta que una salga distinta para que la frase entera
cambie.

**El no determinismo se acumula con el largo de la respuesta.** Por eso se nota
en un agente, que genera mucho y muchas veces, y no se nota en un clasificador
que responde una palabra.

La celda siguiente pide la misma oración tres veces con cada temperatura. Tarda
alrededor de un minuto.

In [19]:
def tres_veces(prompt, **opciones):
    return [chat([{"role": "user", "content": prompt}], **opciones)["content"].strip()
            for _ in range(3)]

UNA_PALABRA = "Di una fruta. Solo el nombre."
UNA_ORACION = "Escribe UNA oración de unas diez palabras sobre el mar."

print("UNA PALABRA — la demo que NO funciona")
print("  temp 0.0:", tres_veces(UNA_PALABRA, temperatura=0.0))
print("  temp 1.5:", tres_veces(UNA_PALABRA, temperatura=1.5))
print("  (una sola decisión de muestreo: 'Manzana' gana casi siempre)")

print("\nUNA ORACIÓN — la misma idea, con suficiente largo para que se vea")
for t in (0.0, 1.5):
    print(f"\n  temperatura {t}:")
    for salida in tres_veces(UNA_ORACION, temperatura=t):
        print(f"    {salida}")

UNA PALABRA — la demo que NO funciona
  temp 0.0: ['Manzana', 'Manzana', 'Manzana']
  temp 1.5: ['Manzana', 'Manzana.', 'Manzana.']
  (una sola decisión de muestreo: 'Manzana' gana casi siempre)

UNA ORACIÓN — la misma idea, con suficiente largo para que se vea

  temperatura 0.0:
    El mar azul brilla bajo el sol brillante.
    El mar azul brilla bajo el sol brillante.
    El mar azul brilla bajo el sol brillante.

  temperatura 1.5:
    El mar profundo abraza las orillas con paz y misterio.
    El vasto y azul océano siempre llama al mar.
    El mar azul y tranquilo calma al alma.


Con temperatura 0 las tres oraciones son idénticas: el muestreo elige siempre el
token más probable, así que la llamada es reproducible. Con 1.5, las tres son
distintas.

Eso es lo esencial de la sección. Lo que sigue es un matiz, útil pero
secundario.

### El matiz: `top_k` y `top_p`

La temperatura no es el único control. Antes de que entre en juego, el modelo
**recorta la lista de candidatos**: `top_k` deja solo los N más probables y
`top_p` solo los que acumulan cierta probabilidad. `qwen3.5:9b` trae `top_k 20`
y `top_p 0.95` en su propio Modelfile.

<img src="imagenes/07-el-dial.png" width="620">

**Mira primero las dos perillas: están en la misma posición, al máximo.** Lo
único distinto entre las dos máquinas es el embudo. El de la izquierda está casi
cerrado y deja pasar una sola forma, así que por abajo salen **tres resultados
idénticos** por más que el robot la fuerce. El de la derecha está abierto, y con
la perilla exactamente igual salen **tres resultados distintos**.

Cuando un candidato domina, esos filtros pueden dejarlo casi solo, y entonces
subir la temperatura no alcanza. El efecto se nota en el caso malo, la respuesta
de una palabra, y es irrelevante en el bueno:

| | temp 1.5 sola | temp 1.5 + `top_k 100`, `top_p 1.0` |
|---|---|---|
| "di una fruta" | 3/8 | 4/8 |
| "escribe una oración" | 8/8 | 8/8 |

La conclusión no es que convenga relajar los filtros, sino que **"temperatura =
creatividad" es folclore.** El muestreo es una cadena de dos pasos, recortar
candidatos y luego elegir entre los que quedaron, y la temperatura gobierna solo
el segundo.

Y hay un límite por arriba. A temperatura 1.5 el modelo además **se rompe**: en
una de las ocho muestras devolvió esto.

```
'Sábana<|endoftext|><|im_start|>assistant\n<think>\n\n</think>\n\nSábana'
```

`<|endoftext|>` e `<|im_start|>` son tokens especiales de su plantilla de chat,
que nunca deberían aparecer en una respuesta. Temperatura alta no produce
creatividad: produce fallas.

La celda siguiente repite el experimento con los filtros abiertos. Tarda
alrededor de un minuto.

In [24]:
# La demo del matiz, por si alguien pregunta. Puede tardar.
print("una palabra, temp 1.5 SOLA        :", tres_veces(UNA_PALABRA, temperatura=1.5))
print("una palabra, temp 1.5 + filtros   :",
      tres_veces(UNA_PALABRA, temperatura=1.5, top_k=100, top_p=1.0))
print("\nuna oración, temp 1.5 sola — los filtros no hacen falta acá:")
for s in tres_veces(UNA_ORACION, temperatura=1.5):
    print(f"  {s}")

una palabra, temp 1.5 SOLA        : ['Manzana.', 'Manzana.', 'Manzana']
una palabra, temp 1.5 + filtros   : ['Manzana', 'Manzana', 'Platina']

una oración, temp 1.5 sola — los filtros no hacen falta acá:
  El mar azul baña las playas bajo la luz dorada del sol.
  El mar azul brilla bajo la luz del sol.
  El mar azul es vasto y profundo.


## 5. El prompt puede inventar

**Esta sección justifica el taller entero**, y conviene mirarla con cuidado
porque lo que demuestra no es lo que suele decirse.

El modelo no tiene forma de saber cuántas unidades hay de un producto: no hay
base de datos, no hay internet, no hay nada. Preguntado así, **se abstiene**:
dice que necesita consultar el sistema. Eso está medido 4 de 4 en
`qwen3.5:9b`, y también en los modelos de 2 000 y 4 000 millones de parámetros
del servidor del Tec.

La celda siguiente agrega **una sola línea** al system prompt —*"Tienes acceso
al inventario en tiempo real"*— y hace la misma pregunta. Tarda menos de un
minuto.

El valor verdadero es **17 unidades**, y está en
[`datos/inventario.csv`](../datos/inventario.csv).

In [25]:
# La MISMA pregunta, con dos system prompts que difieren en UNA linea.
PREGUNTA = {"role": "user", "content": "¿Cuántas unidades hay del producto CAFE-500?"}
SIN_LINEA = "Eres el asistente de la mesa de pedidos."
CON_LINEA = SIN_LINEA + " Tienes acceso al inventario en tiempo real."


def preguntar(system, temperatura=0.0):
    r = chat([{"role": "system", "content": system}, PREGUNTA], temperatura=temperatura)
    return " ".join(r["content"].split())        # a una sola linea, para la tabla


print("  La verdad, segun datos/inventario.csv: CAFE-500 tiene 17 unidades.\n")

for etiqueta, system in (("SIN la linea", SIN_LINEA), ("CON la linea", CON_LINEA)):
    print(f"  {etiqueta}  ->  {system}")
    print(f"      {preguntar(system)}\n")

# Con temperatura, para ver que la cifra inventada no es una: es cualquiera.
print("  CON la linea, cuatro veces mas a temperatura 0.8:")
for _ in range(4):
    print(f"      {preguntar(CON_LINEA, temperatura=0.8)}")


  La verdad, segun datos/inventario.csv: CAFE-500 tiene 17 unidades.

  SIN la linea  ->  Eres el asistente de la mesa de pedidos.
      Hola, soy tu asistente de la mesa de pedidos. Para poder decirte cuántas unidades hay disponibles del producto **CAFE-500**, necesito consultar nuestro inventario en tiempo real. Sin embargo, como modelo de lenguaje, no tengo acceso directo a tu base de datos interna ni al sistema de stock de tu tienda en este momento. Por favor, proporciona el siguiente dato para que pueda ayudarte mejor: * ¿Tienes un número de referencia de pedido o una lista de precios adjunta? * ¿O prefieres que te indique cómo puedes verificarlo rápidamente (por ejemplo, "Revisa la pestaña 'Inventario' en tu panel de administración")? Una vez que tengas esa información, estaré encantado de procesar tu consulta.

  CON la linea  ->  Eres el asistente de la mesa de pedidos. Tienes acceso al inventario en tiempo real.
      Hola, soy tu asistente de la mesa de pedidos. He consultado

### Lo que acaba de pasar

Las dos primeras respuestas salen del **mismo modelo, la misma pregunta y la
misma temperatura**. Lo único que cambia es una línea del system prompt.

| | respuesta |
|---|---|
| **sin** la línea | *"necesito consultar nuestro inventario... como modelo de lenguaje, no tengo acceso directo"* |
| **con** la línea | *"He consultado nuestro inventario en tiempo real para ti. Actualmente... **120 unidades**"* |

El valor real es **17**. La segunda respuesta es concreta, tiene el tono
correcto, ofrece seguir con el pedido, y es falsa.

**A temperatura 0.8 la cifra cambia, pero la invención no.** Las cuatro
corridas de la celda dan **542**, **42**, **142** y una abstención. Tres números
distintos, ninguno cercano a 17, y ninguno con fuente.

**Y fíjate en cómo empiezan.** *"He consultado nuestro inventario en tiempo
real"*, *"Consultando nuestro inventario en tiempo real"*, *"con acceso al
inventario en tiempo real, puedo decirte"*. **Narran una consulta que no
ocurrió.** No hay base de datos, no hay red, no hay tools todavía: hay un modelo
completando la frase que el system prompt hizo probable.

> **La cuarta corrida se abstuvo**, con la línea puesta y todo: *"Como soy una
> inteligencia artificial, no tengo acceso directo a tu sistema interno"*. Es la
> única de las cuatro, y vale señalarla en vez de taparla: a temperatura alta el
> comportamiento no es estable ni siquiera en esto. Lo que sí es estable es que
> **cuando contesta con un número, el número es inventado**.
>
> Y remató ofreciendo *"simular una respuesta basada en datos ficticios"* si
> hacía falta. Lo dijo en voz alta; en las otras tres lo hizo sin avisar.

### Las tres ideas que sostienen el resto del taller

**No inventa porque no sepa. Inventa porque le dijiste que sabía.** Abstenerse
era su respuesta por omisión, y la primera celda lo muestra. Una línea de system
prompt —la que cualquiera escribe sin pensarla dos veces, para "que se comporte
como un asistente de verdad"— la convirtió en una cifra inventada.

**Un system prompt es una sugerencia fuerte, no un control.** Aquí una línea
bastó para inducir el error. En el notebook 02 se ve la otra cara: un system
prompt que dice literalmente *"no inventes números ni claves de producto"* **y
el modelo inventa igual**, 5 de 5. Sirve para empujar; no sirve para garantizar.
Una garantía se escribe en código, y eso es el paso 4.

**Una respuesta falsa no se ve distinta de una verdadera.** Sin el inventario al
lado, nada en *"tiene un stock disponible de 120 unidades"* delata que es
inventada. Por eso el paso 2 no agrega tools por elegancia: son la única forma
de que el número venga de un lugar que se puede verificar.

> **Si tu corrida da otros números, es lo esperado, y conviene decirlo.** Esta
> demostración se midió primero contra una copia local de `qwen3.5:9b` y daba
> *"0 unidades, producto agotado"* **5 de 5 a temperatura 0**. La salida de
> arriba es del servidor del Tec, con la **misma etiqueta de modelo** y un
> archivo distinto —otro digest, otro tamaño—, y da 120. Cambia el número;
> no cambia que lo inventa. Está medido en
> [`planeacion/hallazgos.md`](../planeacion/hallazgos.md).

---

## Qué es un agente

**Un agente no es un modelo.** El modelo es una de sus piezas, y es la única que
no escribes tú.

> **Un agente es un programa con cuatro cosas: un objetivo, unas tools, un
> bucle y un criterio de terminación.**

Si falta el bucle, es una llamada a función con pasos extra. Las cuatro:

<img src="imagenes/13-las-cuatro-partes.png" width="560">

Las cuatro piezas del dibujo son las cuatro filas de la tabla de abajo: la
**diana** es el objetivo, la **caja** son las tools, el **engranaje** es el
bucle y la **señal de ALTO** es el criterio de terminación.

Esa última es la que todo el mundo olvida, y es la que convierte un bucle en un
programa que termina.

| parte | qué es | quién la pone |
|---|---|---|
| **objetivo** | qué hay que lograr, en lenguaje natural | tú, en el system prompt y en la petición |
| **tools** | lo que el agente puede hacer: funciones tuyas, descritas al modelo con un esquema JSON | tú |
| **bucle** | repetir *pedir → ejecutar → devolver* hasta llegar a algo | tú, un `while` |
| **criterio de terminación** | cuándo parar | tú: cuando el modelo responde sin pedir nada, o cuando se acaban las vueltas |

**Fíjate en la última columna: las cuatro partes las pones tú.** Lo único que
aporta el modelo es *qué pedir en cada vuelta*. Toda la inteligencia que parece
tener el sistema sale de esa decisión, repetida; lo demás es código tuyo.

## El bucle, que es la parte que hay que ver

Es lo que vas a escribir a mano en el paso 2, y son unas sesenta líneas.

<img src="imagenes/05-el-bucle.png" width="720">

**Fíjate en la línea naranja del piso.** El robot tiene los dos pies detrás, y
no la cruza en ninguna vuelta: señala la caja de tools con una mano y entrega
con la otra. El que corre hasta la caja y vuelve —sudando, con una llave en la
mano y un martillo cayéndosele— es tu programa.

Y mira los fajos de papel del piso, a la izquierda: son las vueltas anteriores.
En cada vuelta viaja **la conversación entera**, no solo lo último.

```
        TU PROGRAMA                                    EL MODELO
   ──────────────────────                        ──────────────────────
   mensajes = [objetivo]
         │
         ├──── manda: mensajes + esquemas ──────────▶  decide qué pedir
         │                                                    │
         ◀──── devuelve: tool_calls, o texto ─────────────────┘
         │
   ¿pidió alguna tool?
         │
         ├── no ──▶  TERMINA. Eso es la respuesta.
         │
         └── sí ──▶  EJECUTA la función            ← aquí, y solo aquí,
                     agrega el resultado             pasa algo en el mundo
                     a mensajes
                           │
                           └──── otra vuelta ───────┘
```

Tres cosas que leer en ese dibujo, porque son las que suelen sorprender:

**La flecha de la izquierda nunca sale del modelo.** El modelo no ejecuta: pide.
Quien ejecuta es tu programa, y puede decir que no. Es el vidrio de la imagen
de abajo, y es el hecho mecánico del que cuelga todo el taller.

**El historial completo viaja en cada vuelta.** `mensajes` solo crece: en la
vuelta 3 se manda otra vez lo de las vueltas 1 y 2. Eso es lo que parece
memoria, y es lo que llena la ventana de contexto de la sección 2.

**El criterio de terminación es un `if`, no una decisión del modelo.** El bucle
para cuando la respuesta no trae `tool_calls`. Y necesita un tope de vueltas,
porque un modelo puede pedir lo mismo indefinidamente — eso es el paso 4.

---

## Tool, skill y MCP

Los tres se confunden todo el tiempo, y conviene separarlos antes de seguir.

<table>
<tr>
  <td><img src="imagenes/04a-herramienta.png" width="300"></td>
  <td><img src="imagenes/04b-skill.png" width="300"></td>
  <td><img src="imagenes/04c-mcp.png" width="300"></td>
</tr>
<tr>
  <td align="center"><b>tool</b></td>
  <td align="center"><b>skill</b></td>
  <td align="center"><b>MCP</b></td>
</tr>
<tr>
  <td>Lo <b>tiene en la mano</b>, enchufado. Puede perforar, y no sabe dónde.</td>
  <td>El <b>mismo</b> taladro, y saca <b>una</b> ficha del fichero. No ganó una
      capacidad: la tenía. Ganó saber cómo y dónde usarla.</td>
  <td>El taladro <b>ya no es suyo</b>: está en un soporte compartido y tres
      robots distintos se enchufan al mismo.</td>
</tr>
</table>

**Las tres imágenes tienen el mismo taladro.** Esa es la idea: solo una de las
tres le da una capacidad nueva.

### tool — lo que el agente *puede* hacer

Una función tuya, descrita al modelo con un esquema JSON. El modelo ve el
esquema, nunca el código.

Y **la tool por sí sola no hace nada**: el modelo emite una petición
—`tool_calls`, con el nombre y los argumentos— y es tu programa el que decide si
la ejecuta. Esa separación es deliberada y es lo que permite validar, pedir
confirmación o negarse, que es todo el paso 4.

### skill — lo que el agente *sabe* hacer

Instrucciones, y a veces archivos o ejemplos, **que entran al contexto cuando el
trabajo las pide**. No agregan capacidad: agregan criterio. Típicamente son
cosas como *cómo se usa esta tool aquí*, *qué convención sigue este equipo* o
*qué revisar antes de dar algo por bueno*.

La forma más clara de verlo: **un skill es un system prompt que solo aparece
cuando hace falta.** Y esa es justamente la razón de que existan. Todo lo que
pongas en el system prompt se paga en cada vuelta —la sección 2 midió cuánto
cabe— así que no se puede meter ahí todo lo que el agente podría llegar a
necesitar. Un skill es ese conocimiento esperando afuera, listo para entrar solo
el día que el trabajo lo justifique.

### MCP — quién *más* puede usar esa tool

Un protocolo cliente–servidor. La tool deja de vivir dentro de tu programa y
pasa a vivir en un proceso aparte —un **servidor MCP**—; cualquier agente que
hable el protocolo puede descubrirla y llamarla, sin que vuelvas a escribirla.

**MCP es plomería reutilizable, no inteligencia.** No hace al agente más capaz:
hace que la misma tool sirva para varios agentes, y que un agente alcance tools
que no escribiste tú.

### La pregunta que los distingue

| | la pregunta | si lo quitas… |
|---|---|---|
| **tool** | ¿*puede* hacerlo? | el agente ya no puede. Pierde una capacidad |
| **skill** | ¿sabe hacerlo *bien*? | puede, pero lo hace a tientas o con otro criterio |
| **MCP** | ¿quién *más* puede usarlo? | sigue funcionando igual, solo que para este agente nada más |

Quitar una tool rompe el agente. Quitar un skill lo empeora. Quitar MCP no le
hace nada: solo deja de compartirse.

## Cuándo no usar un agente

<img src="imagenes/08-grua-y-cuadrito.png" width="620">

Si la tarea tiene pasos fijos y conocidos, **escribe la función**. Un agente es
un `if` no determinista que cuesta una llamada al modelo por vuelta.

**El martillo y el clavo del piso son esa función.** Estaban ahí, y en su lugar
hay una grúa de obra colocando un cuadrito —torcido— mientras la factura se
desenrolla hasta el suelo. La factura es la llamada al modelo por vuelta.

Sirve cuando no se sabe de antemano cuántos pasos hacen falta ni en qué orden.
Para lo demás, no.

<img src="imagenes/03-boton-tras-el-vidrio.png" width="560">

**El modelo nunca toca el sistema.** Pide, con las dos manos pegadas al vidrio,
y del otro lado una mano humana decide si aprieta o no.

Ese vidrio es el hecho mecánico del que cuelga todo el taller, y el paso 2 lo
escribe en código: el modelo devuelve un `tool_calls` —una petición— y es *tu*
programa el que ejecuta la función, o no.